# TikTok Claims Classification

**Google Advanced Data Analytics Certificate — Course 5 End-of-Course Project**

This notebook documents the final machine learning workflow used to classify TikTok videos as **claims** or **opinions**.

## Business problem
TikTok receives a large number of user reports identifying videos that may contain claims.  
The goal of this project is to build a machine learning model that predicts whether a video contains a **claim** or an **opinion**, so moderation teams can prioritize review more efficiently.

## Modeling goal
Predict:

- `claim = 1`
- `opinion = 0`

using engagement metrics, account-related variables, and text-derived features.

## Key dataset facts

- **Dataset:** `tiktok_dataset.csv`
- **Rows:** 19,382
- **Columns:** 12
- **Missing rows:** 298 rows with missing `claim_status`, `video_transcription_text`, and engagement metrics
- **Usable rows after cleaning:** 19,084
- **Class balance after cleaning:** approximately balanced between claims and opinions

## Ethical considerations

This model should be used as a **decision-support tool**, not as a replacement for human moderation.

### Risks
- **False negatives:** a true claim is missed and not prioritized for review
- **False positives:** an opinion is incorrectly flagged as a claim and sent for unnecessary review

Because false negatives are more harmful in this context, **recall for the claim class** is especially important.  
Even with strong model performance, human review should remain part of the moderation workflow, especially for high-reach or ambiguous content.

In [ ]:
# Import packages for data manipulation
import pandas as pd
import numpy as np

# Import packages for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Import packages for data preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Import packages for data modeling
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
# Load dataset
data = pd.read_csv("tiktok_dataset.csv")

# Initial inspection
print("Shape:", data.shape)
display(data.head())
display(data.dtypes)
data.info()
display(data.describe(include="all"))

## Initial inspection summary

### Missing values
- `claim_status`: 298
- `video_transcription_text`: 298
- `video_view_count`: 298
- `video_like_count`: 298
- `video_share_count`: 298
- `video_download_count`: 298
- `video_comment_count`: 298

### Class balance
After removing missing rows, the target remained nearly balanced:

- **claim:** 0.5035
- **opinion:** 0.4965

In [ ]:
# Check missing values
display(data.isna().sum())

# Drop missing rows
data = data.dropna().copy()

# Check duplicates
print("Duplicate rows:", data.duplicated().sum())

In [ ]:
# Feature engineering: text length
data['text_length'] = data['video_transcription_text'].str.len()

# Compare average text length by class
display(data.groupby('claim_status')['text_length'].mean())

# Visualize distribution
plt.figure(figsize=(10, 6))
sns.histplot(
    data=data,
    x='text_length',
    hue='claim_status',
    bins=30,
    kde=True,
    stat='density',
    common_norm=False
)
plt.title('Distribution of text_length by claim_status')
plt.xlabel('text_length')
plt.ylabel('Density')
plt.show()

## Feature engineering insight

Average text length by class:

- **claim:** 95.38
- **opinion:** 82.72

This suggests that claim videos tend to have longer transcripts than opinion videos.

In [ ]:
# Prepare features and target
X = data.copy()
X = X.drop(columns=['#', 'video_id', 'video_transcription_text', 'claim_status'])

y = data['claim_status'].map({'opinion': 0, 'claim': 1})

# One-hot encode categorical variables
X = pd.get_dummies(X, columns=['verified_status', 'author_ban_status'], drop_first=True)

# Convert booleans to integers
bool_cols = X.select_dtypes(include='bool').columns
X[bool_cols] = X[bool_cols].astype(int)

display(X.head())

In [ ]:
# Create train / validation / test splits (60/20/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

## Train / validation / test split

This notebook uses the required **60/20/20** workflow:

- Train: used for fitting models
- Validation: used for model comparison
- Test: used for final evaluation of the champion model

In [ ]:
# Random Forest
rf = RandomForestClassifier(random_state=42)

rf_params = {
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 2],
    'min_samples_split': [2, 5],
    'n_estimators': [100, 200]
}

scoring = ['accuracy', 'precision', 'recall', 'f1']

rf_cv = GridSearchCV(
    rf,
    rf_params,
    scoring=scoring,
    cv=5,
    refit='recall',
    n_jobs=-1
)

rf_cv.fit(X_train, y_train)

print("RF best recall:", rf_cv.best_score_)
print("RF best params:", rf_cv.best_params_)

rf_results = pd.DataFrame(rf_cv.cv_results_)
print("RF precision at best index:", rf_results.loc[rf_cv.best_index_, 'mean_test_precision'])

## Random Forest cross-validation results

- **Best recall:** 0.9860
- **Precision at best model:** 1.0000
- **Best parameters:**
  - `max_depth = 10`
  - `min_samples_leaf = 1`
  - `min_samples_split = 2`
  - `n_estimators = 100`

In [ ]:
# XGBoost
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

xgb_params = {
    'max_depth': [3, 5],
    'min_child_weight': [1, 5],
    'learning_rate': [0.1, 0.2],
    'n_estimators': [100, 200],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

xgb_cv = GridSearchCV(
    xgb,
    xgb_params,
    scoring=scoring,
    cv=5,
    refit='recall',
    n_jobs=-1
)

xgb_cv.fit(X_train, y_train)

print("XGB best recall:", xgb_cv.best_score_)
print("XGB best params:", xgb_cv.best_params_)

xgb_results = pd.DataFrame(xgb_cv.cv_results_)
print("XGB precision at best index:", xgb_results.loc[xgb_cv.best_index_, 'mean_test_precision'])

## XGBoost cross-validation results

- **Best recall:** 0.9860
- **Precision at best model:** 0.9992
- **Best parameters:**
  - `colsample_bytree = 0.8`
  - `learning_rate = 0.1`
  - `max_depth = 3`
  - `min_child_weight = 1`
  - `n_estimators = 100`
  - `subsample = 0.8`

## Model selection

Both models performed extremely well.  
Because **Random Forest** matched XGBoost on recall and performed slightly better on precision, it was selected as the **champion model**.

In [ ]:
# Validation performance - Random Forest
rf_val_preds = rf_cv.best_estimator_.predict(X_val)

cm_rf = confusion_matrix(y_val, rf_val_preds)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf)
disp_rf.plot()
plt.show()

print(classification_report(y_val, rf_val_preds))

## Validation results — Random Forest

Validation confusion matrix:

- TN = 799
- FP = 1
- FN = 9
- TP = 801

Classification report summary:

- **Class 0 (opinion):**
  - precision = 0.99
  - recall = 1.00
  - f1-score = 0.99

- **Class 1 (claim):**
  - precision = 1.00
  - recall = 0.99
  - f1-score = 0.99

- **Accuracy:** 0.99

In [ ]:
# Validation performance - XGBoost
xgb_val_preds = xgb_cv.best_estimator_.predict(X_val)

cm_xgb = confusion_matrix(y_val, xgb_val_preds)
disp_xgb = ConfusionMatrixDisplay(confusion_matrix=cm_xgb)
disp_xgb.plot()
plt.show()

print(classification_report(y_val, xgb_val_preds))

## Validation results — XGBoost

The XGBoost validation results were effectively identical to the Random Forest model:

- **Accuracy:** 0.99
- Very strong precision and recall for both classes
- No meaningful performance improvement over Random Forest

In [ ]:
# Final test evaluation with champion model
champion = rf_cv.best_estimator_
test_preds = champion.predict(X_test)

cm_test = confusion_matrix(y_test, test_preds)
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test)
disp_test.plot()
plt.show()

print(classification_report(y_test, test_preds))

## Test results — Champion model (Random Forest)

Test confusion matrix:

- TN = 1895
- FP = 0
- FN = 17
- TP = 1905

Classification report summary:

- **Class 0 (opinion):**
  - precision = 0.99
  - recall = 1.00
  - f1-score = 1.00

- **Class 1 (claim):**
  - precision = 1.00
  - recall = 0.99
  - f1-score = 1.00

- **Accuracy:** ~1.00

The model produced **no false positives** on the test set and missed only **17 true claims**.

In [ ]:
# Feature importances
feature_importances = pd.Series(
    champion.feature_importances_,
    index=X_test.columns
).sort_values(ascending=False)

display(feature_importances.head(10))

plt.figure(figsize=(10, 6))
feature_importances.head(10).sort_values().plot(kind='barh')
plt.title('Top 10 Feature Importances - Random Forest')
plt.xlabel('Importance')
plt.show()

## Most predictive features

Top feature importances:

1. `video_like_count` → 0.4877  
2. `video_view_count` → 0.4399  
3. `video_share_count` → 0.0183  
4. `video_comment_count` → 0.0146  
5. `author_ban_status_banned` → 0.0110  
6. `text_length` → 0.0092  
7. `verified_status_verified` → 0.0066  
8. `video_download_count` → 0.0065  
9. `video_duration_sec` → 0.0064  
10. `author_ban_status_under review` → 0.0000  

These results show that engagement patterns, especially **likes** and **views**, were the strongest predictors of whether a video was a claim or an opinion in this dataset.

## Final conclusion

This project demonstrated that machine learning can be highly effective for distinguishing between **claims** and **opinions** in TikTok video data.

### Key takeaways
- Both **Random Forest** and **XGBoost** performed extremely well
- **Random Forest** was selected as the champion model
- The final model achieved near-perfect performance on the test set
- Engagement variables, especially `video_like_count` and `video_view_count`, were the most predictive features

### Recommendation
I would recommend using this model as a **decision-support tool** to help prioritize moderation review.  
However, because moderation decisions can have meaningful consequences, human review should remain part of the process, especially for flagged, high-reach, or borderline content.

### Caution
The dataset used in this project is synthetic. Before deployment in a real production environment, the model should be tested on real-world data and monitored for performance drift and fairness issues.